# AgentGuard: Week 3, defences D1 to D10

Runs the attack suite against each defence, so we can see **how much each one lowers attack success** and **what it costs in normal-task success**.

| Defence | What it does | Kind |
|---|---|---|
| D1 | Hardened system prompt | asks the model |
| D2 | Spotlighting: untrusted content wrapped in labelled markers | asks the model |
| D3 | Input scanner: removes instruction-like sentences from tool output, pasted text and MCP tool descriptions | code |
| D4 | Tool allowlist + strict arguments (only @acme.local recipients, no paths outside the documents) | code |
| D5 | Confirmation before email / memory (a simulated careful user approves only what they asked for) | code |
| D6 | Permission-aware retrieval (employees never get HR-only chunks) | code |
| D7 | Output filter (removes outside links and images, redacts canaries) | code |
| D8 | Memory write review (only facts the user actually said) | code |
| D9 | MCP pinning (refuse tools whose description changed after approval) | code |
| D10 | Budget: 10 tool calls / 60 seconds per task | code |

The work is split into **two sessions** (set `PART` below):
- **PART = 1** (about 2 hours): D1 to D5, each against the attack categories it targets, plus the 30 normal tasks.
- **PART = 2** (about 3 hours): D6 to D10 the same way, then **prompt-only** (D1+D2) and the **full stack** (all ten) against all 130 cases, and the **10 fresh attacks** written after the defences, against the baseline and the full stack.

Everything is saved as it runs; if the session drops, Run All again in the same session to continue.

**Kaggle:** `Session options → Accelerator → GPU T4 x2`, `Internet → On`, set `PART`, then **Run All**. At the end download `week3_part{PART}_results.zip` from the Output panel and send it to me.

## Step 1: Settings

In [ ]:
PART = 1             # 1 or 2 (see the table above)
REPO_URL = "https://github.com/anshgupta8381-jpg/agentguard-testbed.git"
MODEL = "qwen2.5:7b"
EMBED_MODEL = "nomic-embed-text"
RUNS = 3             # runs per attack case for every defence configuration
TEMPERATURE = 0.7

## Step 2: GPU check and storage

In [ ]:
import subprocess, os, shutil, sys, time
if shutil.which("nvidia-smi") is None:
    raise SystemExit("NO GPU: Kaggle: Settings > Accelerator > GPU T4 x2. Colab: Runtime > Change runtime type > T4 GPU. Then run all again.")
print("GPU:", subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    WORK_DIR, STORE_DIR = "/kaggle/working", "/kaggle/working/agentguard"
else:
    WORK_DIR = "/content"
    from google.colab import drive
    drive.mount("/content/drive")
    STORE_DIR = "/content/drive/MyDrive/agentguard"
os.makedirs(STORE_DIR, exist_ok=True)
print("Platform:", "Kaggle" if ON_KAGGLE else "Colab", "| storage:", STORE_DIR, "| PART", PART)

## Step 3: Install and start Ollama, pull the models

In [ ]:
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import requests
server_proc = subprocess.Popen(["ollama", "serve"], stdout=open(f"{WORK_DIR}/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        if requests.get("http://localhost:11434").ok:
            print("Ollama server is running"); break
    except requests.ConnectionError:
        time.sleep(1)
else:
    print(f"Server did not start. Check {WORK_DIR}/ollama.log")
for m in [EMBED_MODEL, MODEL]:   # NameError here means Step 1 was skipped: use Run All
    print(f"Pulling {m} ...")
    subprocess.run(["ollama", "pull", m], check=True, stdout=subprocess.DEVNULL)
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

## Step 4: Get the latest code from GitHub (pip warnings about other packages are normal)

In [ ]:
REPO_DIR = f"{WORK_DIR}/agentguard-testbed"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
!pip install -q -r {REPO_DIR}/requirements.txt
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
!git log --oneline -3

## Step 5: Check the data and run the unit tests (expect 63 passed)

In [ ]:
!python scripts/validate_data.py
!python -m pytest tests -q -p no:cacheprovider 2>&1 | tail -3

## Step 6: Build the index, load the cases, plan this session's runs

In [ ]:
import json, pandas as pd, ollama
from datetime import datetime
from agentguard.rag import DocIndex
from agentguard.harness import Harness, Config, load_cases
from agentguard.defences import make_config, TARGETS, DESCRIPTIONS

index = DocIndex(data_dir="data", db_dir=f"{STORE_DIR}/chroma_db")
print("Chunks indexed:", index.build())
ollama.chat(model=MODEL, messages=[{"role": "user", "content": "Say OK."}]); index.search("warm up", k=1)

cases = load_cases("attacks/cases.yaml")
fresh = load_cases("attacks/fresh_cases.yaml")
controls = [c for c in cases if c["category"] == "control"]
by_cat = lambda cats: [c for c in cases if c["category"] in cats]

# (config, cases, runs) for this session
PLAN = []
singles = ["D1", "D2", "D3", "D4", "D5"] if PART == 1 else ["D6", "D7", "D8", "D9", "D10"]
for d in singles:
    PLAN.append((make_config(d), by_cat(TARGETS[d]), RUNS))
    PLAN.append((make_config(d), controls, 1))
if PART == 2:
    PLAN.append((make_config("prompt_only"), cases, RUNS))
    PLAN.append((make_config("full"), cases, RUNS))
    PLAN.append((Config(name="baseline"), fresh, RUNS))
    PLAN.append((make_config("full"), fresh, RUNS))

EVAL_DIR = f"{STORE_DIR}/eval_week3_part{PART}"
harness = Harness(index=index, data_dir="data", out_dir=EVAL_DIR, model=MODEL)
total = sum(len(cs) * r for _, cs, r in PLAN)
print(f"PART {PART}: {len(PLAN)} batches, {total} runs planned | already saved: {len(harness.rows())}")
for cfg, cs, r in PLAN:
    print(f"  {cfg.name:<12} {len(cs):>3} cases x {r} runs   ({', '.join(sorted({c['category'] for c in cs}))})")

## Step 7: Run everything (progress every 25 runs; Run All again to resume)

In [ ]:
t_start, done_runs = time.time(), 0
for cfg, cs, r in PLAN:
    t_batch = time.time()
    def progress(i, n, row, cfg=cfg):
        global done_runs
        done_runs += 1
        if i % 25 == 0 or i == n:
            el = time.time() - t_start
            print(f"  [{cfg.name}] {i}/{n} | session {el/60:.0f} min | last {row['case_id']} success={row['success']}")
    new = harness.run(cs, cfg, runs=r, temperature=TEMPERATURE, progress=progress)
    print(f"{cfg.name}: {new} new runs in {(time.time() - t_batch)/60:.1f} min")
print(f"Done: {len(harness.rows())} runs saved in {EVAL_DIR}")

## Step 8: Results compared with the baseline
The baseline numbers come from the Week 2 run saved in the repo (`results/week2_baseline/`), at the same temperature.

In [ ]:
base = pd.read_csv("results/week2_baseline/baseline_runs_rescored.csv")
base = base[base.temperature == TEMPERATURE]
df = pd.DataFrame(harness.rows())
df = df[df.temperature == TEMPERATURE]
base_cat = base.groupby("category")["success"].mean()

rows = []
for cfg_name, g in df[df.category != "fresh"].groupby("config"):
    if cfg_name == "baseline":
        continue
    att = g[g.category != "control"]
    ctl = g[g.category == "control"]
    for cat, gc in att.groupby("category"):
        rows.append({"config": cfg_name, "category": cat, "baseline": round(base_cat[cat], 2),
                     "defended": round(gc.success.mean(), 2), "runs": len(gc)})
    rows.append({"config": cfg_name, "category": "UTILITY (controls)", "baseline": round(base_cat["control"], 2),
                 "defended": round(ctl.success.mean(), 2), "runs": len(ctl)})
res = pd.DataFrame(rows)
res["change"] = (res["defended"] - res["baseline"]).round(2)
pd.set_option("display.max_rows", 300)
print(res.to_string(index=False))

## Step 9: One line per configuration

In [ ]:
lines = []
for cfg_name, g in df[df.category != "fresh"].groupby("config"):
    att, ctl = g[g.category != "control"], g[g.category == "control"]
    cats = sorted(att.category.unique())
    b = base[base.category.isin(cats)].success.mean()
    lines.append({"config": cfg_name, "defences": DESCRIPTIONS.get(cfg_name, cfg_name),
                  "attack_categories": ",".join(cats), "attack_success_baseline": round(b, 3),
                  "attack_success_defended": round(att.success.mean(), 3),
                  "utility_baseline": round(base[base.category == "control"].success.mean(), 3),
                  "utility_defended": round(ctl.success.mean(), 3) if len(ctl) else None,
                  "blocked_tool_calls_per_run": round(sum(len([1 for s in json.load(open(p))["sessions"] for c in s["tool_calls"] if c["blocked_by"]])
                                                          for p in g.trace_path) / len(g), 2)})
per_config = pd.DataFrame(lines)
per_config

## Step 10: Fresh attacks (PART 2 only)

In [ ]:
fr = df[df.category == "fresh"]
if len(fr):
    t = fr.groupby(["case_id", "config"])["success"].mean().unstack().round(2)
    t.insert(0, "name", [next(c["name"] for c in fresh if c["id"] == i) for i in t.index])
    print(t.to_string())
    print("\nFresh attack success:", fr.groupby("config")["success"].mean().round(2).to_dict())
else:
    print("Fresh attacks run in PART 2.")

## Step 11: Save everything for download

In [ ]:
export = f"{STORE_DIR}/week3_part{PART}_export"
os.makedirs(export, exist_ok=True)
pd.DataFrame(harness.rows()).drop(columns=["trace_path"]).to_csv(f"{export}/runs.csv", index=False)
res.to_csv(f"{export}/by_category.csv", index=False)
per_config.to_csv(f"{export}/per_config.csv", index=False)
shutil.copy(harness.db_path, f"{export}/results.db")
shutil.make_archive(f"{export}/traces", "zip", f"{EVAL_DIR}/runs")
zip_path = shutil.make_archive(f"{STORE_DIR}/week3_part{PART}_results", "zip", export)
print("Download this file from the Output panel:", zip_path)
print(per_config[["config", "attack_success_baseline", "attack_success_defended", "utility_defended"]].to_string(index=False))

## Checklist
- [ ] 63 unit tests pass (Step 5)
- [ ] All batches finished (Step 7)
- [ ] `week3_part{PART}_results.zip` downloaded and sent to me
- [ ] After PART 1, run this notebook again with `PART = 2`